In [1]:
from DatasetLoading import RepairDatasetLoader
from RayRepresentationEncoding import RayRepresentationEncoder
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import lightning as L
import torch
from lightning.pytorch.loggers import WandbLogger

In [2]:
torch.set_float32_matmul_precision('medium')

In [3]:
dataset_loader = RepairDatasetLoader()

In [4]:
#test_load = np.load(dataset_loader.test_ray_files[0])
#test_ray_colours = torch.from_numpy(test_load['ray_colours'])
#test_piece_rotation = torch.from_numpy(test_load['piece_rotation'])
#test_rays = torch.from_numpy(test_load['rays'])


In [9]:
class NormalPredictionNetwork(nn.Module):
    def __init__(self):
        super().__init__()


        representation_size = 128

        self.representation_encoder = RayRepresentationEncoder(transformer_layers=1, representation_size=representation_size)


        self.head = nn.Sequential(nn.Linear(representation_size, 64),
                                  nn.BatchNorm1d(64),
                                  nn.ReLU(),
                                  nn.Linear(64, 3))


    def forward(self, ray_colours, ray_locations):
        del ray_locations

        x = self.representation_encoder(ray_colours)
        del ray_colours


        x = torch.mean(x, dim=1)
        x = self.head(x)

        return x

class RepairPatternNormalPrediction(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = NormalPredictionNetwork()
        self.lr = 1e-3

    def get_normals(self, rotations):
        batch_size = rotations.shape[0]
        normals = torch.tensor([0, 1, 0], dtype=torch.float32).repeat(batch_size, 1).unsqueeze(2).to(rotations.get_device())
        return torch.matmul(rotations, normals).squeeze(2)

    def training_step(self, batch, batch_idx):
        ray_colours, piece_rotation, ray_locations = batch
        predicted_normals = self.model(ray_colours, ray_locations)
        true_normals = self.get_normals(piece_rotation)
        loss = nn.functional.mse_loss(predicted_normals, true_normals)
        self.log('train_loss', loss)


        # Calculate angular error
        predicted_normals = nn.functional.normalize(predicted_normals, dim=1)
        true_normals = nn.functional.normalize(true_normals, dim=1)
        cos_angles = torch.clamp(torch.sum(predicted_normals * true_normals, dim=1), -1.0, 1.0)
        angles = torch.acos(cos_angles)  # in radians
        angular_error = torch.mean(angles) * (180.0 / np.pi)  # convert to degrees
        self.log('train_angular_error', angular_error)

        del ray_colours, piece_rotation, ray_locations, predicted_normals, true_normals, cos_angles, angles, angular_error
        return loss

    def test_step(self, batch, batch_idx):
        ray_colours, piece_rotation, ray_locations = batch
        predicted_normals = self.model(ray_colours, ray_locations)
        true_normals = self.get_normals(piece_rotation)
        loss = nn.functional.mse_loss(predicted_normals, true_normals)

        self.log('test_loss', loss)

        # Calculate angular error
        predicted_normals = nn.functional.normalize(predicted_normals, dim=1)
        true_normals = nn.functional.normalize(true_normals, dim=1)
        cos_angles = torch.clamp(torch.sum(predicted_normals * true_normals, dim=1), -1.0, 1.0)
        angles = torch.acos(cos_angles)  # in radians
        angular_error = torch.mean(angles) * (180.0 / np.pi)  # convert to degrees
        self.log('test_angular_error', angular_error)

        del ray_colours, piece_rotation, ray_locations, predicted_normals, true_normals, cos_angles, angles, angular_error, loss

    def validation_step(self, batch, batch_idx):
        ray_colours, piece_rotation, ray_locations = batch
        predicted_normals = self.model(ray_colours, ray_locations)
        true_normals = self.get_normals(piece_rotation)
        loss = nn.functional.mse_loss(predicted_normals, true_normals)
        self.log('val_loss', loss)

        # Calculate angular error
        predicted_normals = nn.functional.normalize(predicted_normals, dim=1)
        true_normals = nn.functional.normalize(true_normals, dim=1)
        cos_angles = torch.clamp(torch.sum(predicted_normals * true_normals, dim=1), -1.0, 1.0)
        angles = torch.acos(cos_angles)  # in radians
        angular_error = torch.mean(angles) * (180.0 / np.pi)  # convert to degrees
        self.log('val_angular_error', angular_error)

        del ray_colours, piece_rotation, ray_locations, predicted_normals, true_normals, cos_angles, angles, angular_error, loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        return optimizer

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        ray_colours, piece_rotation, ray_locations = batch
        predicted_normals = self.model(ray_colours, ray_locations)
        return predicted_normals





In [10]:
wandb_logger = WandbLogger()

In [11]:
#train_ray_dataloader, test_ray_dataloader = dataset_loader.get_ray_dataloaders(batch_size=15)
dataset_loader = RepairDatasetLoader(batch_size=12, dataset_type="RayDataset", rays_folder_name="RayPairs1k")

In [12]:
L.seed_everything(42)
model = RepairPatternNormalPrediction()
trainer = L.Trainer(max_epochs=30, logger=wandb_logger, accelerator='gpu', accumulate_grad_batches=5)
trainer.fit(model, datamodule=dataset_loader)

Seed set to 42
/home/enego/Documents/masters/rayRepresntationTesting/RayRepresentationEncoding.py:55: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(transformer_layer, num_layers=transformer_layers)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/enego/Documents/masters/rayRepresntationTesting/.venv/lib/python3.12/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RANK: 0 - 

Epoch 0: 100%|██████████| 2547/2547 [04:33<00:00,  9.32it/s, v_num=4vfn]   
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 2547/2547 [05:09<00:00,  8.22it/s, v_num=4vfn]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 2547/2547 [04:31<00:00,  9.37it/s, v_num=4vfn]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 2547/2547 [04:20<00:00,  9.76it/s, v_num=4vfn]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████| 2547/2547 [04:18<00:00,  9.86it/s, v_num=4vfn]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 5: 100%|██████████| 2547/2547 [04:18<00:00,  9.85it/s, v_num=4vfn]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/enego/Documents/masters/rayRepresntationTesting/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
